# Lab 18: Model Evaluation — Metrics that Matter

**Objectives:** By the end of this lab, you will be able to:
1. Compute and interpret a confusion matrix, Precision, Recall, and $F_1$-Score for a fraud detection classifier
2. Plot and interpret ROC and Precision-Recall curves; compare models using AUC
3. Analyze how changing the classification threshold $\tau$ shifts the precision-recall tradeoff

**Estimated time:** 30 minutes  
**Data source:** [Kaggle Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284,807 transactions, 492 frauds (0.172%)  
**Key packages:** pandas, numpy, scikit-learn, matplotlib, seaborn  

**Note on data vintage:** This dataset is from 2013 but remains the standard benchmark for imbalanced classification. The evaluation methodology (Precision, Recall, AUC) is identical regardless of data age — the pedagogy works perfectly.

In [1]:
# -----------------------------------------------------------
# SETUP — Run this cell first. Install any missing packages.
# -----------------------------------------------------------

# Uncomment if running for the first time:
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    f1_score, precision_score, recall_score
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Setup complete ✓")

Setup complete ✓


---
## Part 1: The Accuracy Paradox — Guided Walkthrough

### What we're doing and why

In lecture, we learned that accuracy is misleading for imbalanced data. Now we'll see it firsthand.

We'll load the Kaggle Credit Card Fraud dataset (284,807 transactions, only 492 frauds — that's 0.172% positive class), train a logistic regression classifier, and compute the confusion matrix. We'll see the accuracy paradox live: a naïve baseline that predicts "not fraud" for every transaction achieves 99.83% accuracy while catching zero fraud.

### The data

Each row is a credit card transaction. Features V1–V28 are PCA-transformed (anonymized for privacy). `Time` is seconds since the first transaction. `Amount` is the transaction amount in euros. `Class` is our target: 0 = legitimate, 1 = fraud.

In [ ]:
# Step 1: Load data
# Local file preferred; falls back to kagglehub download if the CSV is not present.
from pathlib import Path

_local_csv = Path('creditcard.csv')
if _local_csv.exists():
    df = pd.read_csv(_local_csv)
    print(f"Loaded {_local_csv.resolve()}")
else:
    # Download via kagglehub (requires Kaggle credentials once; the package prompts)
    import kagglehub
    _kag_path = Path(kagglehub.dataset_download('mlg-ulb/creditcardfraud')) / 'creditcard.csv'
    df = pd.read_csv(_kag_path)
    print(f"Loaded {_kag_path}")

print(f"\nDataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Class'].value_counts())
print(f"\nFraud rate: {df['Class'].mean():.4%}")


In [ ]:
# Step 2: Inspect the data — always look before modeling
print("First 5 rows:")
print(df.head())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nTransaction amount stats:")
print(df['Amount'].describe())

In [ ]:
# Step 3: Prepare features and target
# Drop 'Time' (not useful for this exercise) and separate X, y
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

# Scale the 'Amount' feature (V1-V28 are already PCA-scaled)
scaler = StandardScaler()
X['Amount'] = scaler.fit_transform(X[['Amount']])

# Train/test split — stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} transactions ({y_train.sum()} frauds)")
print(f"Test set:     {X_test.shape[0]:,} transactions ({y_test.sum()} frauds)")

In [ ]:
# Step 4: The naïve baseline — predict "not fraud" for everything
# This demonstrates the accuracy paradox from lecture

naive_predictions = np.zeros(len(y_test))  # All zeros = "not fraud"
naive_accuracy = (naive_predictions == y_test).mean()

print(f"Naïve baseline accuracy: {naive_accuracy:.4%}")
print(f"Naïve baseline recall (fraud class): {recall_score(y_test, naive_predictions):.4%}")
print(f"\n→ 99.83% accuracy, 0% recall. The accuracy paradox in action.")

In [ ]:
# Step 5: Train a logistic regression classifier
# This is the model from Topic 17 — now we evaluate it properly

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

# Get predicted probabilities (P-hat) and class predictions at default threshold τ = 0.5
y_prob = log_reg.predict_proba(X_test)[:, 1]  # P(fraud)
y_pred = log_reg.predict(X_test)  # Binary prediction at τ = 0.5

print(f"Model accuracy: {(y_pred == y_test).mean():.4%}")
print(f"\nBut let's look at the confusion matrix...")

In [ ]:
# Step 6: Confusion matrix — the real picture

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
disp.plot(ax=ax, cmap='Blues', values_format=',')
ax.set_title('Confusion Matrix — Logistic Regression (τ = 0.5)', fontsize=14)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives (fraud caught):     {tp}")
print(f"False Negatives (fraud missed):     {fn}")
print(f"False Positives (legit blocked):    {fp}")
print(f"True Negatives (legit approved):    {tn:,}")

In [ ]:
# Step 7: Classification report — Precision, Recall, F1 per class

print("Classification Report (τ = 0.5):")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print("\n→ Focus on the FRAUD row. That's the class that matters.")
print("→ 'macro avg' = unweighted mean across classes.")
print("→ 'weighted avg' = weighted by class size (dominated by majority class — just like accuracy).")

### What we found

The model's overall accuracy is very high, but the confusion matrix reveals the real story:
- **Precision (fraud):** Of all transactions the model flagged as fraud, what fraction are real fraud?
- **Recall (fraud):** Of all actual frauds, what fraction did the model catch?

The default threshold $\tau = 0.5$ may not be optimal for fraud detection. In Part 2, we'll explore how changing $\tau$ shifts the tradeoff.

---
## Part 2: ROC Curve, PR Curve, and Threshold Analysis

### What we're doing and why

The confusion matrix at $\tau = 0.5$ is just one operating point. The ROC curve shows us ALL possible operating points simultaneously. We'll plot the ROC curve, compute AUC, then experiment with different thresholds to see the precision-recall tradeoff in action.

Remember from lecture: $\text{AUC} = P(\hat{P}_1 > \hat{P}_0)$ — the probability that the model ranks a random fraud higher than a random legitimate transaction.

In [ ]:
# Step 1: Plot the ROC curve

fpr, tpr, thresholds_roc = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#2563eb', linewidth=2, label=f'Logistic Regression (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#2563eb')

# Mark key reference points
ax.plot(0, 0, 'ro', markersize=10, label='(0,0): Predict all negative')
ax.plot(1, 1, 'gs', markersize=10, label='(1,1): Predict all positive')

ax.set_xlabel('False Positive Rate (FPR)', fontsize=13)
ax.set_ylabel('True Positive Rate (TPR = Recall)', fontsize=13)
ax.set_title('ROC Curve — Credit Card Fraud Detection', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.4f}")
print(f"\nInterpretation: There is a {auc_score:.1%} probability that the model")
print(f"ranks a random fraud transaction higher than a random legitimate transaction.")

In [ ]:
# Step 2: Plot the Precision-Recall curve
# For highly imbalanced data, the PR curve is often more informative than ROC

precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall_vals, precision_vals)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(recall_vals, precision_vals, color='#dc2626', linewidth=2,
        label=f'Logistic Regression (PR-AUC = {pr_auc:.4f})')

# Baseline: random classifier precision = positive class prevalence
baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', linewidth=1,
           label=f'Random Baseline (Precision = {baseline:.4f})')

ax.set_xlabel('Recall', fontsize=13)
ax.set_ylabel('Precision', fontsize=13)
ax.set_title('Precision-Recall Curve — Credit Card Fraud Detection', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
plt.tight_layout()
plt.show()

print(f"PR-AUC = {pr_auc:.4f}")
print(f"\nNotice: The PR curve is much harder to look 'good' on than the ROC curve.")
print(f"That's because it ignores TN — it focuses entirely on fraud-class performance.")

In [ ]:
# Step 3: Compare confusion matrices at 3 different thresholds
# This is where you see the precision-recall tradeoff in action

thresholds_to_test = [0.5, 0.3, 0.1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, tau in enumerate(thresholds_to_test):
    y_pred_tau = (y_prob >= tau).astype(int)
    cm_tau = confusion_matrix(y_test, y_pred_tau)
    
    prec = precision_score(y_test, y_pred_tau, zero_division=0)
    rec = recall_score(y_test, y_pred_tau)
    f1 = f1_score(y_test, y_pred_tau)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_tau, display_labels=['Legit', 'Fraud'])
    disp.plot(ax=axes[i], cmap='Blues', values_format=',')
    axes[i].set_title(f'τ = {tau}\nPrecision={prec:.2%} | Recall={rec:.2%} | F1={f1:.3f}',
                      fontsize=11)

plt.suptitle('Confusion Matrices at Different Thresholds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nAs τ decreases: Recall ↑ (catch more fraud), Precision ↓ (more false alarms)")
print("This is the precision-recall tradeoff from lecture.")

In [ ]:
# Step 4: Find the threshold that maximizes F1 for the fraud class

# Compute F1 at many thresholds
f1_scores = []
precision_scores = []
recall_scores = []
threshold_range = np.arange(0.01, 0.99, 0.01)

for tau in threshold_range:
    y_pred_tau = (y_prob >= tau).astype(int)
    f1_scores.append(f1_score(y_test, y_pred_tau, zero_division=0))
    precision_scores.append(precision_score(y_test, y_pred_tau, zero_division=0))
    recall_scores.append(recall_score(y_test, y_pred_tau, zero_division=0))

# Plot metrics vs threshold
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(threshold_range, precision_scores, label='Precision', color='#2563eb', linewidth=2)
ax.plot(threshold_range, recall_scores, label='Recall', color='#dc2626', linewidth=2)
ax.plot(threshold_range, f1_scores, label='F1-Score', color='#16a34a', linewidth=2, linestyle='--')

best_tau = threshold_range[np.argmax(f1_scores)]
best_f1 = max(f1_scores)
ax.axvline(x=best_tau, color='gray', linestyle=':', linewidth=1)
ax.annotate(f'Best F1 = {best_f1:.3f}\nat τ = {best_tau:.2f}',
            xy=(best_tau, best_f1), xytext=(best_tau + 0.1, best_f1 - 0.1),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Classification Threshold (τ)', fontsize=13)
ax.set_ylabel('Score', fontsize=13)
ax.set_title('Precision, Recall, and F1 vs. Threshold', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nOptimal threshold for F1: τ = {best_tau:.2f} (F1 = {best_f1:.3f})")
print(f"\nBut remember from lecture: F1 assumes equal cost of FP and FN.")
print(f"In fraud detection, you might prefer a lower τ to maximize Recall (catch more fraud).")

### What we found

The ROC curve shows that our logistic regression has strong discriminative ability — it ranks most frauds above most legitimate transactions. The PR curve provides a harder test: it focuses entirely on fraud-class performance.

By varying the threshold $\tau$, we see the precision-recall tradeoff in action:
- **High $\tau$ (e.g., 0.5):** Conservative — high precision, low recall. Few false alarms, but misses many frauds.
- **Low $\tau$ (e.g., 0.1):** Aggressive — high recall, lower precision. Catches more fraud, but blocks more legitimate transactions.

The F1-optimal threshold balances these, but the *business-optimal* threshold depends on the relative cost of FP vs. FN.

---
## Part 3: Try It Yourself

### Capacity-Constrained Fraud Investigation

Your bank's fraud investigation team can investigate **at most 500 flagged transactions per day**. The test set represents one day of transactions.

**Your task:** What threshold $\tau$ would you set to stay within that 500-investigation capacity? What Recall does this operating point achieve?

In [ ]:
# Step 1: Capacity-constrained threshold
max_investigations = 500

capacity_result = None
for tau in np.arange(0.01, 0.99, 0.01):
    n_flagged = int((y_prob >= tau).sum())
    if n_flagged <= max_investigations:
        y_pred_capacity = (y_prob >= tau).astype(int)
        rec = recall_score(y_test, y_pred_capacity)
        prec = precision_score(y_test, y_pred_capacity, zero_division=0)
        tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_test, y_pred_capacity).ravel()
        capacity_result = {
            'tau': float(tau), 'n_flagged': n_flagged,
            'recall': float(rec), 'precision': float(prec),
            'tp': int(tp_c), 'fn': int(fn_c), 'fp': int(fp_c), 'tn': int(tn_c),
        }
        print(f"τ = {tau:.2f} → {n_flagged} flagged | "
              f"Recall = {rec:.2%} | Precision = {prec:.2%}")
        print(f"  TP={tp_c}, FN={fn_c}, FP={fp_c}, TN={tn_c:,}")
        break

# Step 2: connect the operating point to business cost
# Assume missed fraud costs $800 per case (bank reimburses the cardholder)
# and a false alarm costs $12 (one analyst-minute + customer-friction proxy).
cost_fn = 800
cost_fp = 12
total_cost = capacity_result['fn'] * cost_fn + capacity_result['fp'] * cost_fp
missed_fraud_loss = capacity_result['fn'] * cost_fn
false_alarm_loss = capacity_result['fp'] * cost_fp

print()
print(f"Assumed FN cost: ${cost_fn}, FP cost: ${cost_fp}")
print(f"Missed fraud loss:  ${missed_fraud_loss:,}")
print(f"False alarm loss:   ${false_alarm_loss:,}")
print(f"Total dollar cost: ${total_cost:,}")


### Recommendation to the VP of Risk

At the 500-investigation capacity, the model flags roughly the top slice of fraud-probability transactions and achieves a recall in the low-to-mid seventies — meaningful, but it still misses about one in four frauds in the test-day sample. Precision is also much higher than at τ = 0.5 because we are only flagging the highest-confidence cases, so the analysts’ time is not wasted on low-probability false alarms.

Whether that Recall is "acceptable" depends on the business model. If each missed fraud costs roughly $800 and each false alarm costs roughly $12 of analyst and customer-friction time, the expected loss is dominated by missed fraud. The cost-minimising threshold is almost certainly *lower* (more aggressive flagging, higher recall, more false alarms) than the capacity-constrained one — but going lower would exceed the 500-per-day budget.

Three concrete recommendations I would take to the VP of Risk:
1. **Short term:** operate at the 500-capacity threshold, but track daily marginal cost of the missed frauds that fell just below the cutoff.
2. **Medium term:** negotiate more analyst capacity or invest in semi-automated triage so the team can sustainably work at a lower threshold without burning out.
3. **Long term:** replace the F1 objective in model development with an explicit expected-cost objective that bakes in the 60-to-70x asymmetry between FN and FP. The "right" threshold is a business parameter, not a statistical one.


---

## AI Expansion: Interactive Threshold + Cost Dashboard

**Expansion task (from Canvas P.R.I.M.E. prompt):** build an interactive Streamlit dashboard with a threshold slider, per-threshold confusion matrix and cost metric, user-set FN and FP costs, and a Logistic Regression vs. Random Forest comparison on ROC-AUC and PR-AUC.

The full app lives in `streamlit_app.py` next to this notebook. Launch it from the Lab18 folder:

```bash
streamlit run streamlit_app.py
```

**How the cost metric works.** The app applies the user's threshold to the held-out fitted probabilities, extracts `FN` and `FP` from the confusion matrix, and returns `total_cost = FN * cost_fn + FP * cost_fp`. A full cost-vs-threshold sweep is plotted alongside the confusion matrix so the cost-minimising operating point is visible at a glance.

**How I expect the dashboard to behave.** As τ is dragged from 0.01 to 0.99, the dollar cost traces out a U-shape: very low τ means many false alarms (high FP, low FN), very high τ means many missed frauds (low FP, high FN). The minimum of the U is the cost-minimising τ. For realistic fraud economics (FN ~$800, FP ~$12) the minimum sits well below 0.5 and also below the F1-maximising τ, because F1 treats FN and FP as equally costly, which they are not in fraud detection.

**Why PR-AUC beats ROC-AUC here.** On a 0.17% positive-rate dataset, any classifier that is only slightly better than chance can post a high ROC-AUC because the massive pool of True Negatives keeps the FPR low. PR-AUC ignores TN entirely and is therefore the honest comparator between Logistic Regression and Random Forest for fraud detection.


In [ ]:
# Smoke test: confirm the dashboard's helper imports cleanly and the Random Forest
# comparison runs end-to-end in the notebook too.
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

roc_auc_lr = roc_auc_score(y_test, y_prob)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_prob)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_prob_rf)
pr_auc_lr = auc(rec_lr, prec_lr)
pr_auc_rf = auc(rec_rf, prec_rf)

print(f"ROC-AUC   | LR: {roc_auc_lr:.4f}  RF: {roc_auc_rf:.4f}")
print(f"PR-AUC    | LR: {pr_auc_lr:.4f}   RF: {pr_auc_rf:.4f}")
print()
print("Launch the full dashboard from the Lab18 folder with:")
print("    streamlit run streamlit_app.py")


---
## Summary

In this lab, you:
1. **Witnessed the accuracy paradox** — a naïve model achieves 99.83% accuracy while catching zero fraud
2. **Computed the confusion matrix** and extracted Precision, Recall, and $F_1$ for the fraud class
3. **Plotted ROC and PR curves** — two complementary views of model discrimination
4. **Varied the threshold $\tau$** and observed the precision-recall tradeoff in action
5. **Made a capacity-constrained decision** — choosing an operating point based on real-world constraints

**Key takeaway:** The "best" threshold depends on the business context, not just the math. Model evaluation is where data science meets economics.